# NAO Project: Linear Modelling in the Pooling Framework

## Package Imports

In [2]:
!pip install ftfy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.8 MB/s eta 0:00:00


In [3]:
!pip install mord

  Preparing metadata (setup.py) ... done
  Created wheel for mord: filename=mord-0.7-py3-none-any.whl size=9885 sha256=e418cc546e0905b50f50bd1bca7618c54447fb2a5497a4d249887083444187e0
  Stored in directory: /root/.cache/pip/wheels/d1/fc/57/f2a2ad4ed0491ab6d5bb8642a90f1da9469397641e914743da
Successfully built mord


In [4]:
import kagglehub
from ftfy import fix_text
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import mord
import re
from textblob import TextBlob
import matplotlib.dates as mdates
import warnings
warnings.filterwarnings('ignore')

## Data Summary

We consider the [`skytrax-airline-reviews`](https://www.kaggle.com/datasets/efehandanisman/skytrax-airline-reviews) from Kaggle. After downloading and cleaning the data, we obtain the following cleaned data:

In [9]:
# Download latest version
path = kagglehub.dataset_download("efehandanisman/skytrax-airline-reviews")

# Define the full path to the Excel file
excel_file = os.path.join(path, "capstone_airline_reviews3.xlsx")
survey_df = pd.read_excel(excel_file)
# Dataset Duplicate Value Count
survey_df.duplicated(keep = 'first').sum()
#Dropping the Empty rows
survey_df.drop_duplicates(keep=False,inplace= True)
survey_df.reset_index(inplace=True,drop=True)
survey_df['customer_review'] = survey_df['customer_review'].apply(fix_text)
survey_df = survey_df[survey_df.isna().sum(axis=1) <= 3] # remove entries
survey_df.to_csv('capstone_airline_reviews3_cleaned.csv', index=False)
survey_df = pd.read_csv("capstone_airline_reviews3_cleaned.csv", encoding="utf-8-sig")
survey_df.head()

100%|██████████| 23.1M/23.1M [00:00<00:00, 162MB/s]

Extracting files...


,airline,overall,author,review_date,customer_review,aircraft,traveller_type,cabin,route,date_flown,seat_comfort,cabin_service,food_bev,entertainment,ground_service,value_for_money,recommended
0,Turkish Airlines,7.0,Christopher Hackley,8th May 2019,✅ Trip Verified | London to Izmir via Istanbul...,NaN,Business,Economy Class,London to Izmir via Istanbul,2019-05-01 00:00:00,4.0,5.0,4.0,4.0,2.0,4.0,yes
1,Turkish Airlines,2.0,Adriana Pisoi,7th May 2019,✅ Trip Verified | Istanbul to Bucharest. We ma...,NaN,Family Leisure,Economy Class,Istanbul to Bucharest,2019-05-01 00:00:00,4.0,1.0,1.0,1.0,1.0,1.0,no
2,Turkish Airlines,3.0,M Galerko,7th May 2019,✅ Trip Verified | Rome to Prishtina via Istanb...,NaN,Business,Economy Class,Rome to Prishtina via Istanbul,2019-05-01 00:00:00,1.0,4.0,1.0,3.0,1.0,2.0,no
3,Turkish Airlines,10.0,Zeshan Shah,6th May 2019,✅ Trip Verified | Flew on Turkish Airlines IAD...,A330,Solo Leisure,Economy Class,Washington Dulles to Karachi,April 2019,4.0,5.0,5.0,5.0,5.0,5.0,yes
4,Turkish Airlines,1.0,Pooja Jain,6th May 2019,✅ Trip Verified | Mumbai to Dublin via Istanbu...,NaN,Solo Leisure,Economy Class,Mumbai to Dublin via Istanbul,2019-05-01 00:00:00,1.0,1.0,1.0,1.0,1.0,1.0,no


#### $X$ (explanatory) variables

**Flight Information:**
- `airline`: Which airline the flight was taken (Data type: `categorical` or `text`)
- `review_data`: The date the review was taken. (Data type: `date-time`)
- `customer_review`: The contents of the review. (Data type: `text`)
- `aircraft`: The aircraft of the flight. (Data type: `categorical` or `text`)
- `traveller_type`: The type of traveller (e.g. Business, Family Leisure, Solo Leisure). (Data type: `categorical`)
- `cabin`: The class of the undertaken flight (e.g. Economy, Business and First Class). (Data type: `categorical` or `ordinal` - can possibly be considered ordinal since Economy < Business < First Class).
- `route`: The route of the flight. (Data type: `categorical`)

**Questions with Likert-Scale Responses:**
- `data_flown`: The date of the flight taken. (Data type: `date-time`)
- `seat_comfort`: Likert-scale response to a question on seat comfort. (Data type: `ordinal`)
- `cabin_service`: Likert-scale response to a question on cabin service. (Data type: `ordinal`)
- `food_bev`: Likert-scale response to a question on food/beverage quality. (Data type: `ordinal`)
- `entertainment`: Likert-scale response to a question on entertainment. (Data type: `ordinal`)
- `ground_service`: Likert-scale response to a question on ground service. (Data type: `ordinal`)
- `value_for_money`: Likert-scale response to a question on value for money. (Data type: `ordinal`)

Note that we have ignored the `author` column, since this corresponds to a unique identifier of the row (i.e. equivalent to row ID).


#### $y$ (response) variables

- `overall`: Response to overall rating of experience of flight. (Data type: `ordinal` or possibly `continuous`)
- `recommended`: Response to whether the airline would be recommended. (Data type: `categorical`)


## Questions to Answer

We are interested in how the $y$ (response) variables are influenced by the $X$ (explanatory variables). Examples of questions we would like to answer:

- Do different questions have different $y$ effects?
- Does some question's rating influence $y$ more than another question?
- Is question rating predictive in general across all questions?
- Are some questions better at discriminating $y$ responses?
- Is the effect of a question’s rating constant across subgroups (e.g., economy vs business)?

The advantage of a **Linear Model** is that all of these questions can be framed as **hypothesis tests** on the estimated regression coefficients.

## Simplifying Assumption

#### Response Variable

For the remainder of this notebook, we only consider the `recommended` variables as our single $y$ response variable.
In general, we could:

1. **Ordinal Regression:** Consider only `overall` as the response variable within an ordinal regression model.
2. **Both Response Variables:** Consider both `recommended` and `overall` together within a [General Linear Model](https://en.wikipedia.org/wiki/General_linear_model).

In [14]:
survey_df['recommended'] = survey_df['recommended'].map({'yes': 1, 'no': 0}) # do this once


#### Explanatory Variables

In addition, we drop:
 - The `customer_review` column as an $X$ (explanatory) variable, since it would have to be dealt with separately (e.g. via a text embedding like *sentiment scores* or *transformer-based embeddings*).
 - The `route` column as an explanatory variable, since there aren't many repeated routes (running `survey_df['route'].nunique()` we obtain 24028 unique items). We could infer the length of a flight from this data and use this instead.
 - The `review_date` and `date_flown` columns - just to avoid `date-time` data.
 - The 'aircraft' column - a lot of `NaN`s present.

Let's drop all these columns (including `author`):

In [15]:
survey_df = survey_df.drop(['author', 'customer_review', 'route', 'review_date', 'date_flown', 'aircraft'], axis=1)
survey_df.head()

,airline,overall,traveller_type,cabin,seat_comfort,cabin_service,food_bev,entertainment,ground_service,value_for_money,recommended
0,Turkish Airlines,7.0,Business,Economy Class,4.0,5.0,4.0,4.0,2.0,4.0,1
1,Turkish Airlines,2.0,Family Leisure,Economy Class,4.0,1.0,1.0,1.0,1.0,1.0,0
2,Turkish Airlines,3.0,Business,Economy Class,1.0,4.0,1.0,3.0,1.0,2.0,0
3,Turkish Airlines,10.0,Solo Leisure,Economy Class,4.0,5.0,5.0,5.0,5.0,5.0,1
4,Turkish Airlines,1.0,Solo Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,0


#### Dropping all NaNs

To get a clean dataset, we drop all `NaN` values from the dataset. This avoids numerical issues in the subsequent modelling steps. Alternatively, we could impute the values.

In [16]:
survey_df = survey_df.dropna()
survey_df

,airline,overall,traveller_type,cabin,seat_comfort,cabin_service,food_bev,entertainment,ground_service,value_for_money,recommended
0,Turkish Airlines,7.0,Business,Economy Class,4.0,5.0,4.0,4.0,2.0,4.0,1
1,Turkish Airlines,2.0,Family Leisure,Economy Class,4.0,1.0,1.0,1.0,1.0,1.0,0
2,Turkish Airlines,3.0,Business,Economy Class,1.0,4.0,1.0,3.0,1.0,2.0,0
3,Turkish Airlines,10.0,Solo Leisure,Economy Class,4.0,5.0,5.0,5.0,5.0,5.0,1
4,Turkish Airlines,1.0,Solo Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...
34265,Ukraine International,1.0,Family Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,0
34266,Ukraine International,1.0,Family Leisure,Economy Class,1.0,2.0,1.0,1.0,1.0,1.0,0
34268,Ukraine International,10.0,Couple Leisure,Economy Class,5.0,5.0,5.0,4.0,4.0,4.0,1
34269,Ukraine International,2.0,Solo Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,0


#### Modelling Assumption

Since our response variable is the binary column `recommended`, we model it using *logistic regression*. Specifically, we assume:

$$
Y \mid X = x \sim \mathrm{Bernoulli}(\mathrm{logit}^{-1}(f(x))),
$$

where $f(x)$ is a linear model of the covariates.


## Unpooled Model

In the unpooled model, we treat each row independently and include all explanatory variables directly in the regression. We fit the following linear model for $f(x)$:

$$
f(x) = \beta^\top x,
$$

where $x$ represents a single row of the dataset, including both numerical and encoded categorical variables.

Since several explanatory variables are categorical — such as `airline`, `traveller_type`, and `cabin` — we expand them using indicator variables. The full logistic regression model becomes:

$$
\text{logit}(\Pr(Y \mid X = x)) = \beta_0
+ \sum_{i=1}^{n_a} \alpha_i \, \mathbb{1}(\text{airline} = i)
+ \sum_{i=1}^{n_t} \tau_i \, \mathbb{1}(\text{traveller\_type} = i)
+ \sum_{i=1}^{n_c} \gamma_i \, \mathbb{1}(\text{cabin} = i)
+ \sum_{j=1}^{6} \beta_j \, x_j,
$$

where the last sum runs over the six Likert-scale variables: `seat_comfort`, `cabin_service`, `food_bev`, `entertainment`, `ground_service`, and `value_for_money`.

This can be achieved using the following code:

In [17]:
import statsmodels.formula.api as smf

model = smf.logit(
    formula="""
    recommended ~ C(airline) + C(traveller_type) + C(cabin)
        + seat_comfort + cabin_service + food_bev
        + entertainment + ground_service + value_for_money
    """,
    data=survey_df
)

result = model.fit()
print(result.summary())


         Current function value: 0.148688
         Iterations: 35
                           Logit Regression Results                           
Dep. Variable:            recommended   No. Observations:                21091
Model:                          Logit   Df Residuals:                    21004
Method:                           MLE   Df Model:                           86
Date:                Thu, 01 May 2025   Pseudo R-squ.:                  0.7853
Time:                        21:48:51   Log-Likelihood:                -3136.0
converged:                      False   LL-Null:                       -14609.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                 -10.1538      0.558    -18.202      0.000   

/usr/local/lib/python3.11/dist-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Unfortunately, there is a numerical issue due to one of the airlines (`QantasLink` - with only four total flights) having all of their `recommended` values equalling $1$:

In [18]:
airline_outcome_counts = (
    survey_df
    .groupby('airline')['recommended']
    .value_counts()
    .unstack(fill_value=0)
)

print(airline_outcome_counts.sort_values(by=0))  # Sort by # of "no"s


recommended            0    1
airline                      
QantasLink             0    4
Thai Smile Airways     1   11
Germanwings            4    2
Adria Airways          4    5
Tunisair               5    5
...                  ...  ...
Emirates             491  566
Spirit Airlines      516   32
British Airways      782  475
United Airlines      987  305
American Airlines   1061  272

[75 rows x 2 columns]


To avoid, this we could either drop the whole column or remove rows who have a very small number of flights. Let's do the latter:

In [19]:
# Drop airlines with fewer than 10 rows or only one class in recommended
valid_airlines = (
    airline_outcome_counts
    [(airline_outcome_counts.sum(axis=1) >= 10) &  # enough total
     (airline_outcome_counts[0] > 0) &             # at least one "no"
     (airline_outcome_counts[1] > 0)]              # at least one "yes"
).index

survey_df = survey_df[survey_df['airline'].isin(valid_airlines)]
survey_df

,airline,overall,traveller_type,cabin,seat_comfort,cabin_service,food_bev,entertainment,ground_service,value_for_money,recommended
0,Turkish Airlines,7.0,Business,Economy Class,4.0,5.0,4.0,4.0,2.0,4.0,1
1,Turkish Airlines,2.0,Family Leisure,Economy Class,4.0,1.0,1.0,1.0,1.0,1.0,0
2,Turkish Airlines,3.0,Business,Economy Class,1.0,4.0,1.0,3.0,1.0,2.0,0
3,Turkish Airlines,10.0,Solo Leisure,Economy Class,4.0,5.0,5.0,5.0,5.0,5.0,1
4,Turkish Airlines,1.0,Solo Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,0
...,...,...,...,...,...,...,...,...,...,...,...
34265,Ukraine International,1.0,Family Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,0
34266,Ukraine International,1.0,Family Leisure,Economy Class,1.0,2.0,1.0,1.0,1.0,1.0,0
34268,Ukraine International,10.0,Couple Leisure,Economy Class,5.0,5.0,5.0,4.0,4.0,4.0,1
34269,Ukraine International,2.0,Solo Leisure,Economy Class,1.0,1.0,1.0,1.0,1.0,1.0,0


Rerunning the model:

In [20]:
import statsmodels.formula.api as smf

model = smf.logit(
    formula="""
    recommended ~ C(airline) + C(traveller_type) + C(cabin)
        + seat_comfort + cabin_service + food_bev
        + entertainment + ground_service + value_for_money
    """,
    data=survey_df
)

result = model.fit()
print(result.summary())


Optimization terminated successfully.
         Current function value: 0.148770
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:            recommended   No. Observations:                21072
Model:                          Logit   Df Residuals:                    20988
Method:                           MLE   Df Model:                           83
Date:                Thu, 01 May 2025   Pseudo R-squ.:                  0.7852
Time:                        21:49:06   Log-Likelihood:                -3134.9
converged:                       True   LL-Null:                       -14596.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                              coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------------------------------
Intercept                                 -10.1501

### Evaluating Unpooled Approach: Likert-Scale Variables Only

Due to the large number of different airlines, let's just drop everything apart from the Likert Scale questions. So, our model is:

$$
\text{logit}(\Pr(Y \mid X = x)) = \beta_0  + \sum_{j=1}^{6} \beta_j \, x_j.
$$


In [21]:
cleaned_df = survey_df.drop(['airline', 'traveller_type', 'cabin'], axis=1)
cleaned_df

,overall,seat_comfort,cabin_service,food_bev,entertainment,ground_service,value_for_money,recommended
0,7.0,4.0,5.0,4.0,4.0,2.0,4.0,1
1,2.0,4.0,1.0,1.0,1.0,1.0,1.0,0
2,3.0,1.0,4.0,1.0,3.0,1.0,2.0,0
3,10.0,4.0,5.0,5.0,5.0,5.0,5.0,1
4,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0
...,...,...,...,...,...,...,...,...
34265,1.0,1.0,1.0,1.0,1.0,1.0,1.0,0
34266,1.0,1.0,2.0,1.0,1.0,1.0,1.0,0
34268,10.0,5.0,5.0,5.0,4.0,4.0,4.0,1
34269,2.0,1.0,1.0,1.0,1.0,1.0,1.0,0


In [22]:
import statsmodels.formula.api as smf

model = smf.logit(
    formula="""
    recommended ~  seat_comfort + cabin_service + food_bev
        + entertainment + ground_service + value_for_money
    """,
    data=cleaned_df
)

result = model.fit()
print(result.summary())


Optimization terminated successfully.
         Current function value: 0.152815
         Iterations 9
                           Logit Regression Results                           
Dep. Variable:            recommended   No. Observations:                21072
Model:                          Logit   Df Residuals:                    21065
Method:                           MLE   Df Model:                            6
Date:                Thu, 01 May 2025   Pseudo R-squ.:                  0.7794
Time:                        21:49:21   Log-Likelihood:                -3220.1
converged:                       True   LL-Null:                       -14596.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                      coef    std err          z      P>|z|      [0.025      0.975]
-----------------------------------------------------------------------------------
Intercept         -11.0829      0.183    -60.676      0.000     -11.441     -10.725
seat_comfort  

#### Interpretation

All six Likert-scale predictors are statistically significant (p < 0.001). Their coefficients represent the log-odds change of recommending the airline per 1-point increase in rating:

| Predictor         | Coef | Interpretation (holding others constant)                                                |
|-------------------|------|-------------------------------------------------------------------------------------------|
| **Intercept**     | –11.30 | Base log-odds when all ratings are zero (all the other rating increases must overcome the -11.30 value) |
| **Seat Comfort**  | +0.438 | Each unit increase in seat comfort increases log-odds of recommending by 0.438 (~55% increase in odds) |
| **Cabin Service** | +0.551 | Very strong effect — increases odds by ~73% per unit                                  |
| **Food & Bev**    | +0.398 | Moderate effect — increases odds by ~49% per unit                                     |
| **Entertainment** | +0.177 | Small but significant effect (~19% increase per unit)                                 |
| **Ground Service**| +0.676 | Strong effect — increases odds by ~97% per unit                                       |
| **Value for Money**| **+1.389** | **Largest effect** — each unit increase multiplies odds by ~4 (exp(1.389))              |

#### Limitation of Unpooled Modelling

The unpooled (“wide-format”) approach suffers from several limitations, especially when it comes to hypothesis testing and scalability to a higher number of question-answer variables. For example:

1. **No natural framework for comparing slopes**  
   - We need to fit separate coefficients for each question-rating variable (e.g. `seat_comfort`, `food_bev`, etc.), then manually construct and test contrasts (e.g. “Is the effect of `seat_comfort` significantly larger than that of `food_bev`?”).  
   - In contrast, a *pooled model* can evaluate these comparisons directly via $H_0: \beta_{\text{food\_bev}} - \beta_{\text{seat\_comfort}} = 0$.

2. **Inflexible handling of missing data**  
   - Any row with a missing rating for *any* question must be dropped (or imputed separately for each variable), leading to variable sample sizes per coefficient.  
   - The *pooled framework* allows you to drop (or model) missingness at the question-response level, preserving more data.

3. **Scalability of number of parameters**  
   - As the number of question-answer variables grows, the number of predictors in the unpooled model grows linearly (one coefficient per question). Testing interactions with other covariates (e.g. question × traveller_type) becomes combinatorially large.  
   - The pooled model keeps a compact design: one factor for `Question_ID` plus one rating variable (with interaction), regardless of how many questions you add.

4. **Lack of partial pooling/shrinkage**  
   - Unpooled estimates are “no pooling” — each coefficient stands alone, even if some questions have few responses. This can lead to high variance and overfitting.  
   - A multilevel or even a fixed-effect pooled model borrows strength across questions, reducing bias and variance in estimates for sparse categories (questions with a few responses).

5. **Hard to infer group-level effects**  
   - To ask “Do cabin-service ratings matter more for Business travellers than Family travellers?” you’d need to add and test a three-way interaction for each question separately.  
   - In the pooled framework, you simply include `C(Question_ID) * Rating * C(traveller_type)` and perform joint tests on the interaction terms.

## Pooling Framework

To perform pooling, we introduce a `Question_ID` and `rating` $X$ (explanatory variable). This is sometimes called rewriting the data-frame in "long-format".

In [24]:
likert_cols = [
    'seat_comfort',
    'cabin_service',
    'food_bev',
    'entertainment',
    'ground_service',
    'value_for_money'
]

cleaned_df_long = pd.melt(
    survey_df,
    id_vars=['recommended','overall'],    # keep recommended with each row
    value_vars=likert_cols,     # turn these columns into rows
    var_name='Question_ID',        # name for the former-column labels
    value_name='rating'         # name for the former-cell values
)

cleaned_df_long


,recommended,overall,Question_ID,rating
0,1,7.0,seat_comfort,4.0
1,0,2.0,seat_comfort,4.0
2,0,3.0,seat_comfort,1.0
3,1,10.0,seat_comfort,4.0
4,0,1.0,seat_comfort,1.0
...,...,...,...,...
126427,0,1.0,value_for_money,1.0
126428,0,1.0,value_for_money,1.0
126429,1,10.0,value_for_money,4.0
126430,0,2.0,value_for_money,1.0


Since our outcome remains the binary `recommended`, in the pooled (long-format) framework we model each question–response pair $(Q, R)$ as

$$ Y \mid Q = q,\;R = r
\;\sim\;\mathrm{Bernoulli}\!\bigl(\mathrm{logit}^{-1}\bigl(f(q,r)\bigr)\bigr), $$

where the linear predictor is

$$ f(q, r)
= \beta_{0} + \beta_1 \, r
+ \sum_{j=1}^{6}\alpha_{j}\,\mathbf{1}(Q=j)
+ \sum_{j=1}^{6}\delta_{j}\,\mathbf{1}(Q=j)\,r. $$

Equivalently:

$$
\begin{align*}
\text{logit}\bigl(\Pr(Y=1\mid Q,R)\bigr)
&= \beta_{0} + \beta_{1} \,R
+ \sum_{j=1}^{6}\alpha_{j}\,\mathbf{1}(Q=j)
+ \sum_{j=1}^{6}\delta_{j}\,\mathbf{1}(Q=j)\,R \\
&= \underbrace{\beta_{0} + \sum_{j=1}^{6}\alpha_{j}\,\mathbf{1}(Q=j)}_{\text{question‐specific intercepts}}
\;+\;
\underbrace{\beta_{1}\,R}_{\text{global slope}}
\;+\;
\underbrace{\sum_{j=1}^{6}\delta_{j}\,\mathbf{1}(Q=j)\,R}_{\text{question‐specific deviations in slope}}.
\end{align*}
$$

Here:

- $\alpha_{j}$ captures the question-specific intercept shift for question $j$.  
- $\delta_{j}$ allows the slope of the rating $R_i$ to vary by question.  

This single model lets us directly compare both baseline effects $\alpha_j$ and rating effects $\delta_j$ across all questions.  


This can be implemented using the following code:

In [25]:
model = smf.logit(
    formula="""
    recommended ~  C(Question_ID) + C(Question_ID) * rating
    """,
    data=cleaned_df_long
)

result = model.fit()
print(result.summary())


Optimization terminated successfully.
         Current function value: 0.349947
         Iterations 8
                           Logit Regression Results                           
Dep. Variable:            recommended   No. Observations:               126432
Model:                          Logit   Df Residuals:                   126420
Method:                           MLE   Df Model:                           11
Date:                Thu, 01 May 2025   Pseudo R-squ.:                  0.4948
Time:                        21:49:58   Log-Likelihood:                -44245.
converged:                       True   LL-Null:                       -87574.
Covariance Type:            nonrobust   LLR p-value:                     0.000
                                               coef    std err          z      P>|z|      [0.025      0.975]
------------------------------------------------------------------------------------------------------------
Intercept                                   -5.6

#### Interpretation

##### 1. Baseline (Reference)—**Cabin Service**  
- **Intercept** = –5.5916:
  This is the log-odds of “recommended = 1” when `Question_ID = cabin_service` and `rating = 0`.  (Rating = 0 is outside the 1–5 scale, so this intercept is just the mathematical anchor.)  
- **rating** = +1.6039:
  For **cabin_service**, each one-point increase in rating multiplies the odds of recommending by $\exp(1.6039)\approx4.97$.  

##### 2. Question-specific intercept shifts  
Each of these tells you how the baseline log-odds (at rating = 0) for that question differs from `cabin_service`’s intercept:

| Question             | Coef        | Interpretation                                                       |
|----------------------|-------------|------------------------------------------------------------------------|
| entertainment        | +2.0481     | At rating=0, entertainment starts ~7.8× higher odds than cabin_service ($\exp(2.0481)\approx7.75$).  |
| food_bev             | +1.0546     | food_bev starts ~2.87× higher odds.                                    |
| ground_service       | +1.0751     | ground_service starts ~2.93× higher odds.                              |
| seat_comfort         | +0.3097     | seat_comfort starts ~1.36× higher odds.                                |
| value_for_money      | –1.8162     | value_for_money starts ~0.16× the odds (much lower baseline).          |


##### 3. Question-specific deviations in slope  
These interaction terms show how each question’s *rating slope* differs from the **baseline slope** of 1.6039:

| Question             | Interaction | Slope (total)                   | Odds-ratio per unit rating                |
|----------------------|-------------|----------------------------------|-------------------------------------------|
| cabin_service        | —           | **1.6039**                       | $\exp(1.6039)\approx4.97$               |
| entertainment        | –0.4166     | 1.6039 – 0.4166 = **1.1873**      | $\exp(1.1873)\approx3.28$               |
| food_bev             | –0.0538     | 1.6039 – 0.0538 = **1.5501**      | $\exp(1.5501)\approx4.71$               |
| ground_service       | –0.1154     | 1.6039 – 0.1154 = **1.4885**      | $\exp(1.4885)\approx4.43$               |
| seat_comfort         | +0.0651     | 1.6039 + 0.0651 = **1.6690**      | $\exp(1.6690)\approx5.31$               |
| value_for_money      | +0.7460     | 1.6039 + 0.7460 = **2.3499**      | $\exp(2.3499)\approx10.48$              |

To summarise:

1. **All coefficients are highly significant** (p < 0.05), so none of these effects are due to chance.  
2. **Value for Money** has by far the steepest slope: a one-point increase multiplies the odds of recommending by over $10\times$, compared to $\approx 5\times$ for the average question.  
3. **Seat Comfort** and **food & beverage** also have slopes above the baseline, indicating they’re slightly more predictive than cabin service.  
4. **Entertainment** has the weakest rating effect ($\approx3.3\times$ odds per point), despite its high intercept shift—meaning people start out more likely to recommend after answering the entertainment question, but additional points matter less.  

Altogether, the output confirms that **not all questions are equally predictive**: some dimensions (especially value for money) drive recommendation far more strongly than others.

### Comparison of Pooled vs. Unpooled

- The unpooled model tells you how important each question is on its own, but leaves you to do all inter-question comparisons by hand.

- The pooled model gives you both a global sense of how ratings drive recommendation and—via the interaction terms—exactly how and by how much each question deviates from that global effect, all within one coherent hypothesis-testing framework.

For example, here is a concrete hypothesis you can answer directly from the pooled model output—but cannot without extra work in the unpooled (“wide”) model:

> **Question:** Is the effect of “value_for_money” ratings on the odds of recommending the airline significantly larger than the average rating effect across all questions?

#####  Pooled model

From the summary, we obtained:

- `rating`: with coefficient $\beta_1 = 1.6039$. This is the "global" slope - the *average* effect across questions.
- `C(Question_ID)[T.value_for_money]:rating`: with coefficeint $\delta_{\text{vfm}} = 0.7460$. This is the question-specific *slope deviation* for `value_for_money`.

To answer the question above, we perform the hypothesis:

$$ H_0:\;\delta_{\text{vfm}} = 0
\quad\Longleftrightarrow\quad
\underbrace{\beta_1 + \delta_{\text{vfm}}}_{\text{slope for vfm}}
\;=\;
\underbrace{\beta_1}_{\text{average slope}} $$

The obtained p-value for `C(Question_ID)[T.value_for_money]:rating` directly tests whether the `value_for_money` slope differs from the *average* slope.  Here, $\delta_{\text{vfm}}=0.7460$ with $p<0.001$, so we conclude:

> **The rating effect of value_for_money is significantly larger than the average rating effect across all questions.**

#### Why the unpooled model can’t answer that directly

In the unpooled (“wide”) model we obtained get six separate slopes $\hat\beta_{\text{seat}},\dots,\hat\beta_{\text{vfm}}$.  But there is **no “global” slope** in the model to compare against.  To test

\[
\hat\beta_{\text{vfm}}
\;>\;\frac{1}{6}\bigl(\hat\beta_{\text{seat}} + \cdots + \hat\beta_{\text{vfm}}\bigr),
\]

you’d have to:

1. **Compute** the six estimated coefficients,
2. **Average** them,
3. **Manually compute** the variance of that average (using the joint covariance matrix of the six estimates),
4. **Form** a \(t\)- or Wald–statistic,
5. **Look up** its p-value.

That process is **tedious, error-prone**, and not directly supported by a single line of output.  

---

### 3. Take-away

- **Pooled model** gives you a **global slope** and **per-question deviations** with **direct p-values** for “deviation = 0” (i.e.\ “does this question’s slope differ from the global average?”).  
- **Unpooled model** lacks a global parameter, so comparing any individual coefficient to the overall mean requires manual contrasts.

This is why **pooling** unlocks hypothesis tests about **“how each question’s effect relates to the overall effect”** in a single, coherent framework.

In [31]:
import statsmodels.api as sm
from statsmodels.miscmodels.ordinal_model import OrderedModel

# 1. Prepare the formula and data
formula = 'overall ~  C(Question_ID) + C(Question_ID) * rating'

# 2. Create and fit the ordinal model (logit link)
model_ordinal = OrderedModel.from_formula(
    formula,
    data=cleaned_df_long,
    distr='logit'  # Use 'probit' for ordered probit
)
result_ordinal = model_ordinal.fit(method='bfgs')  # Optimization method

# 3. Print results
print(result_ordinal.summary())

Optimization terminated successfully.
         Current function value: 1.665710
         Iterations: 78
         Function evaluations: 81
         Gradient evaluations: 81
                             OrderedModel Results                             
Dep. Variable:                overall   Log-Likelihood:            -2.1060e+05
Model:                   OrderedModel   AIC:                         4.212e+05
Method:            Maximum Likelihood   BIC:                         4.214e+05
Date:                Thu, 01 May 2025                                         
Time:                        21:59:02                                         
No. Observations:              126432                                         
Df Residuals:                  126412                                         
Df Model:                          11                                         
                                               coef    std err          z      P>|z|      [0.025      0.975]
--------

### Interpretation of the Ordered Logistic Regression Results

#### **Model Overview**
- **Dependent Variable**: `overall` (ordinal rating, likely 1–10).
- **Method**: Maximum Likelihood Estimation (converged successfully).
- **Observations**: 126,432 reviews.
- **Key Metrics**:
  - **Log-Likelihood**: -210,600 (baseline for model comparison).
  - **AIC/BIC**: 421,200/421,400 (used for model selection).

### **Key Findings**
#### **1. Main Effects (Question Categories)**
- **Entertainment**: Significant positive effect (`coef=0.7742, p<0.001`).  
  - Passengers who rated "entertainment" higher had 2.17x higher odds (`exp(0.7742)`) of a better overall rating.  
- **Food & Beverage**: Positive effect (`coef=0.5218, p<0.001`).  
- **Ground Service**: Positive effect (`coef=0.5119, p<0.001`).  
- **Seat Comfort** and **Value for Money**: Not significant (p=0.889 and p=0.230).

#### **2. Interaction Effects (Question × Rating)**
- **Entertainment × Rating**: Negative interaction (`coef=-0.0758, p<0.001`).  
  - The positive effect of entertainment diminishes as the rating increases.  
- **Seat Comfort × Rating**: Strong positive interaction (`coef=0.1206, p<0.001`).  
  - Seat comfort becomes more influential at higher ratings.  
- **Value for Money × Rating**: Positive interaction (`coef=0.1354, p<0.001`).  
  - Value for money gains importance as ratings increase.

#### **3. Rating Variable**
- **Main Effect**: Strong positive coefficient (`coef=1.4618, p<0.001`).  
  - A 1-unit increase in the predictor variable `rating` (likely an aggregated score) leads to a 1.46-unit increase in the log-odds of a higher overall rating.  

#### **4. Cutpoints (Thresholds Between Ratings)**
- **1.0/2.0**: Large threshold (`coef=2.6145`), indicating a steep jump from the lowest rating (1) to the next (2).  
- **9.0/10.0**: Positive threshold (`coef=0.2502`), suggesting higher ratings are more easily distinguished.  
- **Mid-range thresholds (e.g., 7.0/8.0)**: Negative coefficients, reflecting natural clustering of mid-tier ratings.


### **Business Implications**
1. **Entertainment and Food Drive Satisfaction**:  
   - Focus on improving in-flight entertainment and meal quality for immediate impact.  
2. **Seat Comfort Matters at High Ratings**:  
   - Luxury/priority services should emphasize seat comfort to retain high-tier customers.  
3. **Value for Money Gains Importance**:  
   - Budget airlines should highlight cost-effectiveness to boost overall ratings.  


### **Actionable Recommendations**
- **Target Entertainment and Food Quality**:  
  - Partner with content providers for better in-flight entertainment.  
  - Upgrade meal options for premium cabins.  
- **Optimize Seat Comfort for High-Rating Customers**:  
  - Offer seat upgrades or premium seating options.  
- **Leverage Value for Money**:  
  - Promote competitive pricing and loyalty rewards for budget-conscious travelers.  
